## Resultados Analisis de SBOMs

En este analisis, se revisarán los repositorios publicos de Fastlane, una plataforma de codigo abierto para simplificar el desarrollo de apps para Android y iOS

In [ ]:
import subprocess
import sys

# Instalar el proyecto con todas sus dependencias
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

Primero se instalan las dependencias 'python-dotenv' y 'requests', para clonar los repositorios usando un token de github

Ejecute este script para obtener los 8 repositorios de Fastlane

```uv run scripts/generate_repo_list.py```

In [ ]:
import os
import requests
import json
import subprocess
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()

# Configuración
ORG_NAME = "fastlane"
TOKEN = os.getenv("GITHUB_TOKEN")


def get_active_repos(org):
    url = f"https://api.github.com/orgs/{org}/repos"
    params = {"per_page": 100, "sort": "pushed"}

    # Se añade el token a los headers para evitar límites de la API de GitHub
    headers = {"Authorization": f"token {TOKEN}"} if TOKEN else {}
    response = requests.get(url, params=params, headers=headers)

    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        return []

    repos = response.json()

    # Calcular la fecha de hace un mes
    one_month_ago = datetime.now() - timedelta(days=30)
    repo_list = []

    for r in repos:
        # Filtrar por fecha de último push (commit)
        pushed_at = datetime.strptime(r['pushed_at'], "%Y-%m-%dT%H:%M:%SZ")

        if pushed_at > one_month_ago:
            repo_list.append({
                # Usamos clone_url (termina en .git) que es ideal para clonar
                "url": r['clone_url'],
                "path": f"data/repos/{r['name']}",
                "ref": r['default_branch']
            })

    return repo_list


# 1. Asegurar que los directorios existan
os.makedirs("./data/repos", exist_ok=True)

# 2. Generar la estructura final y buscar los repositorios
active_repos = get_active_repos(ORG_NAME)
output = {"repositories": active_repos}

# 3. Guardar en archivo JSON en la nueva ruta
json_path = "./data/repos.json"
with open(json_path, 'w') as f:
    json.dump(output, f, indent=4)

print(f"Se han encontrado {len(active_repos)} repositorios activos.")
print(f"Lista guardada exitosamente en: {json_path}")

# 4. Clonar los repositorios hacia data/repos
print("\n--- Iniciando clonación de repositorios ---")
for repo in active_repos:
    repo_url = repo["url"]
    repo_path = repo["path"]

    # Comprobar si la carpeta ya existe para no intentar clonarlo dos veces
    if not os.path.exists(repo_path):
        print(f"Clonando {repo_url} en {repo_path}...")
        # Ejecuta el comando git clone en la terminal de forma invisible
        subprocess.run(["git", "clone", repo_url, repo_path])
    else:
        print(f"El repositorio ya existe en {repo_path}. Omitiendo...")

print("\n¡Proceso finalizado! Ya tienes tus repositorios listos para generar el SBOM.")

Los repositorios analizados son:
- codesigning.guide (Principalmente archivos HTML y assets estáticos para la guía de firma de código. No contiene código ejecutable con dependencias gestionadas.)
- docker (Repositorio de configuración Docker con pipelines de CI/CD usando GitHub Actions para automatizar builds y deployments.)
- docs (Proyecto de documentación generada con Jekyll/similar. Las dependencias son principalmente herramientas de build y procesamiento de contenido en Ruby.)
- fastlane (Núcleo del ecosistema. Contiene la lógica principal de automatización para aplicaciones iOS y Android, con dependencias en múltiples lenguajes.)
- fastlane-sirp (Repositorio de extensión/plugin para Fastlane con configuración de CI/CD minimalista.)
- fastlane.tools (Repositorio del sitio web oficial de Fastlane (fastlane.tools). Utiliza acciones personalizadas de GitHub para el setup y construcción.)
- frameit-frames (Biblioteca de frames para screenshots en Fastlane. Configuración mínima con Node.js para procesamiento.)
- github-actions (Colección de GitHub Actions reutilizables para el ecosistema. Dependencias principalmente en el toolkit oficial de GitHub Actions.)

#### Resultados de SBOMs de los 8 repositorios

Estadisticas:
- Repositorios analizados: 8
- Lenguajes identificados: Ruby, Swift, JavaScript, YAML, Cocoa/iOS
- Repositorios con dependencias: 7
- Repositorios sin dependencias: 1 (codesigning.guide)

Componentes detectados por repositorio:
- codesigning.guide: 0 dependencias (recursos estáticos)
- docker: GitHub Actions para CI/CD (checkout v6, docker/build-push-action v7)
- docs: 100+ gemas Ruby para documentación (CFPropertyList, addressable, etc.)
- fastlane: 200+ dependencias mixtas (Ruby, Swift, Cocoa/iOS)
- fastlane-sirp: GitHub Actions minimalista (checkout v4)
- fastlane.tools: Acciones personalizadas de GitHub
- frameit-frames: GitHub Actions con Node.js (checkout v6, setup-node v6)
- github-actions: 50+ paquetes NPM (@actions/core, @actions/exec, @actions/http-client)

Conclusiones
- Arquitectura modular y multilingüe con especialización por repositorio
- Automatización robusta mediante GitHub Actions
- Dependencias estables con versiones pinned
- Seguridad auditable mediante SBOMs para identificar vulnerabilidades

#### Resultados de CodeQL de los 8 repositorios

Estadísticas:
- Repositorios analizados: 8
- Total de problemas detectados: 11 (únicamente en fastlane.tools)
- Repositorios sin problemas: 7
- Severidad de problemas: Todas warnings (sin errores críticos)

Problemas detectados por repositorio:
- codesigning.guide: 0 problemas
- docker: 0 problemas
- docs: 0 problemas
- fastlane: Análisis completo sin reportes críticos
- fastlane-sirp: 0 problemas
- fastlane.tools: 11 warnings (scripts sin integridad + variables no utilizadas)
- frameit-frames: 0 problemas
- github-actions: 0 problemas

Hallazgos principales:
- Scripts desde CDN sin verificación de integridad (2 warnings) - fastlane.tools/views/base.html
- Variables locales no utilizadas (9 warnings) - archivos JavaScript en fastlane.tools
- No se detectaron vulnerabilidades de seguridad críticas
- Ausencia de inyecciones SQL, XSS, CSRF o desserialización insegura

Conclusiones:
- Ecosistema Fastlane mantiene estándares altos de seguridad
- Riesgos identificados son de baja severidad (warnings de calidad de código)
- Recomendación: Agregar atributos integrity a scripts CDN en fastlane.tools
- Continuar con análisis periódicos de CodeQL en CI/CD

#### Resultados de Grype de los 8 repositorios

Estadísticas:
- Repositorios analizados: 8
- Total de vulnerabilidades detectadas: 57
- Repositorios sin vulnerabilidades: 3
- Severidad máxima detectada: Baja (0 críticas, 0 altas, 0 medias)

Vulnerabilidades detectadas por repositorio:
- codesigning.guide: Sin dependencias gestionadas
- docker: Sin dependencias gestionadas
- docs: 6 vulnerabilidades bajas
- fastlane: 16 vulnerabilidades bajas
- fastlane-sirp: 0 vulnerabilidades ✓
- fastlane.tools: 35 vulnerabilidades bajas
- frameit-frames: 0 vulnerabilidades ✓
- github-actions: 0 vulnerabilidades ✓

Vulnerabilidades principales:

docs (6 vulnerabilidades):
- Jinja2: Sandbox breakout attacks (3 CVEs)
- Markdown: Uncaught Exception (GHSA-5wmx-573v-2qwq)
- Addressable: ReDoS en templates (GHSA-h27x-rffw-24p4)
- PyMdown Extensions: ReDOS bug (GHSA-r6h4-mm7h-8pmq)

fastlane (16 vulnerabilidades):
- Sinatra: Vulnerabilidad de confianza y ReDoS (2 CVEs)
- WebRick: HTTP Request/Response Smuggling (2 CVEs)
- Rack: Unbounded file uploads y quadratic complexity (2 CVEs)
- YARD: Cross-site Scripting en frames.html
- Y 9 vulnerabilidades adicionales

fastlane.tools (35 vulnerabilidades):
- GitPython: Remote Code Execution (2 CVEs)
- Werkzeug: Debugger RCE y safe_join issues (3 CVEs)
- urllib3: Cookie header leak en redirects
- Jinja2: HTML attribute injection
- Y 29 vulnerabilidades adicionales

Conclusiones:
- Estado general: SEGURO (sin vulnerabilidades críticas)
- Todas las vulnerabilidades son de severidad baja
- Repositorios más comprometidos: fastlane.tools (35), fastlane (16), docs (6)
- Repositorios limpios: fastlane-sirp, frameit-frames, github-actions
- Problemas principales: Dependencias web antiguas (Werkzeug 1.0.1, urllib3 1.26.5)
- Recomendación: Actualizar stack web en fastlane.tools y fastlane a versiones modernas